# Notebook 03 — Exploratory Analysis

Runs all analysis queries against the SQLite database and produces visualizations.
Findings from this notebook feed directly into Notebook 04.

In [ ]:
import sys
sys.path.insert(0, '..')

import sqlite3
from pathlib import Path

import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt

from src.data_cleaning import get_connection
from src.visualization import (
    plot_funding_over_time,
    plot_top_funders,
    plot_state_choropleth,
    plot_grant_size_distribution,
    plot_ntee_breakdown,
    plot_pre_post_2020,
)

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

DB = Path('../data/racial_equity_grants.sqlite')
conn = get_connection(DB)

PROC = Path('../data/processed')

In [ ]:
# Load cleaned DataFrames for visualization helpers
grants_df = pd.read_sql('SELECT * FROM grants', conn)
re_grants  = grants_df[grants_df['is_racial_equity'] == 1].copy()
print(f"Total grants: {len(grants_df):,} | Racial equity: {len(re_grants):,}")

## 1  Summary Statistics

In [ ]:
summary = pd.read_sql("""
    SELECT
        COUNT(*) AS total_grants,
        COUNT(DISTINCT funder_ein) AS unique_funders,
        COUNT(DISTINCT recipient_ein) AS unique_recipients,
        ROUND(SUM(grant_amount)/1e6, 1) AS total_dollars_m,
        ROUND(AVG(grant_amount)) AS avg_grant_size,
        MIN(tax_year) AS earliest_year,
        MAX(tax_year) AS latest_year
    FROM grants
    WHERE is_racial_equity = 1
""", conn)
summary.T.rename(columns={0: 'value'})

## 2  Time Series: Racial Equity Funding by Year

In [ ]:
annual = pd.read_sql("""
    SELECT tax_year,
           COUNT(*) AS grant_count,
           ROUND(SUM(grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants
    WHERE is_racial_equity = 1 AND tax_year IS NOT NULL
    GROUP BY tax_year ORDER BY tax_year
""", conn)
annual

In [ ]:
fig = plot_funding_over_time(re_grants)
fig.savefig(PROC / 'fig_funding_over_time.png', bbox_inches='tight')
plt.show()

## 3  Pre- vs. Post-2020 Statistical Test

We use a two-sample t-test on annual grant totals to test whether the mean
annual funding level changed significantly after 2020.

In [ ]:
pre  = annual[annual['tax_year'] < 2020]['total_dollars_m']
post = annual[annual['tax_year'] >= 2020]['total_dollars_m']

t_stat, p_value = stats.ttest_ind(pre, post, equal_var=False)  # Welch's t-test

print(f"Pre-2020  mean: ${pre.mean():.1f}M/yr  (n={len(pre)} years)")
print(f"Post-2020 mean: ${post.mean():.1f}M/yr  (n={len(post)} years)")
print(f"Welch's t = {t_stat:.2f}, p = {p_value:.4f}")
print("Significant at α=0.05?" , 'Yes' if p_value < 0.05 else 'No')

In [ ]:
fig = plot_pre_post_2020(re_grants)
fig.savefig(PROC / 'fig_pre_post_2020.png', bbox_inches='tight')
plt.show()

## 4  Top 20 Funders

In [ ]:
top_funders = pd.read_sql("""
    SELECT g.funder_ein,
           COALESCE(f.name, g.funder_ein) AS funder_name,
           f.state AS funder_state,
           COUNT(*) AS grant_count,
           ROUND(SUM(g.grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants g
    LEFT JOIN foundations f ON g.funder_ein = f.ein
    WHERE g.is_racial_equity = 1
    GROUP BY g.funder_ein
    ORDER BY SUM(g.grant_amount) DESC
    LIMIT 20
""", conn)
top_funders

In [ ]:
fig = plot_top_funders(re_grants)
fig.savefig(PROC / 'fig_top_funders.png', bbox_inches='tight')
plt.show()

## 5  Geographic Distribution

In [ ]:
by_state = pd.read_sql("""
    SELECT recipient_state,
           COUNT(*) AS grant_count,
           ROUND(SUM(grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants
    WHERE is_racial_equity = 1 AND recipient_state IS NOT NULL
    GROUP BY recipient_state
    ORDER BY SUM(grant_amount) DESC
""", conn)
by_state.head(10)

In [ ]:
try:
    fig_map = plot_state_choropleth(re_grants)
    fig_map.write_html(str(PROC / 'fig_state_map.html'))
    fig_map.show()
except ImportError as e:
    print(e)

## 6  Grant Size Distribution

In [ ]:
fig = plot_grant_size_distribution(re_grants)
fig.savefig(PROC / 'fig_grant_size_dist.png', bbox_inches='tight')
plt.show()

## 7  NTEE Category Breakdown

In [ ]:
ntee_data = pd.read_sql("""
    SELECT r.ntee_major,
           COUNT(g.id) AS grant_count,
           ROUND(SUM(g.grant_amount)/1e6, 2) AS total_dollars_m
    FROM grants g
    JOIN recipients r ON g.recipient_ein = r.ein
    WHERE g.is_racial_equity = 1 AND r.ntee_major IS NOT NULL
    GROUP BY r.ntee_major
    ORDER BY SUM(g.grant_amount) DESC
""", conn)
ntee_data

In [ ]:
if len(ntee_data):
    re_grants_ntee = re_grants.merge(
        pd.read_sql('SELECT ein, ntee_major FROM recipients', conn),
        left_on='recipient_ein', right_on='ein', how='left'
    ).dropna(subset=['ntee_major'])
    fig = plot_ntee_breakdown(re_grants_ntee)
    fig.savefig(PROC / 'fig_ntee.png', bbox_inches='tight')
    plt.show()

## 8  Funder Concentration (Herfindahl Index)

A Herfindahl-Hirschman Index (HHI) > 0.25 indicates high concentration.

In [ ]:
funder_shares = re_grants.groupby('funder_ein')['grant_amount'].sum()
total = funder_shares.sum()
hhi = ((funder_shares / total) ** 2).sum()
print(f"HHI (funder concentration): {hhi:.4f}")
print(f"Top 5 funders account for {(funder_shares.nlargest(5).sum()/total):.1%} of total dollars")

Proceed to **Notebook 04** to write up findings with narrative context.